In [ ]:
import pandas as pd
from pprint import pprint as pp

import os

Last run: 2026-03-14 07:32:39


In [13]:
%load_ext autoreload
%autoreload 2

from obituary_parser import parse_pairs_nonbatch, parse_pairs_parallel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Last run: 2026-03-14 07:32:39


In [14]:
RUN_MODE = "parallel"  # "parallel" or "nonbatch"
CHUNK_SIZE = 1000
MAX_ROWS = None  # set e.g. 200 for quick tests; keep None for full dataset
MAX_WORKERS = max(1, (os.cpu_count() or 1) - 1)

Last run: 2026-03-14 07:32:39


In [15]:
df = pd.read_csv('obituaris_6_FINAL.csv')
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   url        40000 non-null  str  
 1   data       40000 non-null  str  
 2   biografia  40000 non-null  str  
dtypes: str(3)
memory usage: 937.6 KB


,url,data,biografia
0,https://www.currentobituary.com/obit/107571,03/12/2012,Mrs. Lillian A. Ferullo of Medford died on Mon...
1,https://www.currentobituary.com/obit/107566,03/12/2012,"Cole, Opal Irene (Gadd), 84, passed away March..."
2,https://www.currentobituary.com/obit/107563,03/12/2012,"CHARLTON: Donald S. Lapierre, 75, of 17 Reynol..."
3,https://www.currentobituary.com/obit/107559,03/12/2012,"FRAMINGHAM: Frances I. “Fran” (Moore) Carroll,..."
4,https://www.currentobituary.com/obit/107553,03/12/2012,"Blythe, GA---Mrs. Mildred L. Godwin, 74, enter..."


Last run: 2026-03-14 07:32:39


In [62]:
source_series

,biografia,data
0,Mrs. Lillian A. Ferullo of Medford died on Mon...,03/12/2012
1,"Cole, Opal Irene (Gadd), 84, passed away March...",03/12/2012
2,"CHARLTON: Donald S. Lapierre, 75, of 17 Reynol...",03/12/2012
3,"FRAMINGHAM: Frances I. “Fran” (Moore) Carroll,...",03/12/2012
4,"Blythe, GA---Mrs. Mildred L. Godwin, 74, enter...",03/12/2012
...,...,...
39995,"Mr Gerard G. Bonenfant, of Taunton, died Thurs...",07/02/2009
39996,"Mrs. Obie D. Nelms, 85, of Fitzgerald, died, T...",07/02/2009
39997,New Bedford-Edward Mendonca age 86 years died ...,07/02/2009
39998,"Boston –Dorothy L. (Correia) Garcia, age 83 ye...",07/02/2009


Last run: 2026-03-14 08:02:35


In [246]:
MAX_ROWS =  None
source_df = df[['data', 'biografia']] if MAX_ROWS is None else df[['data', 'biografia']].iloc[:MAX_ROWS]
source_series = source_df.apply(lambda row: (row['data'], row['biografia']), axis=1)
pairs = list(source_series.items())

if RUN_MODE == "nonbatch":
    parse_data = parse_pairs_nonbatch(pairs)
else:
    parse_data = parse_pairs_parallel(
        pairs,
        chunk_size=CHUNK_SIZE,
        max_workers=MAX_WORKERS,
    )

Last run: 2026-03-14 09:34:30


In [275]:
# parse_data: {row_id: {'sections': {...}, 'extracted': {...}}}
result = pd.DataFrame.from_dict(parse_data, orient='index').sort_index()
result['id'] = result.index

extracted_df = result['extracted'].apply(pd.Series)
sections_df = result['sections'].apply(pd.Series).replace({'': pd.NA})

result = pd.concat([result.drop(columns=['extracted', 'sections']), extracted_df, sections_df], axis=1)

Last run: 2026-03-14 09:54:14


In [293]:
max_obit_vs_death_days_diff = 20
max_birth_vs_death_years_diff = 150

result['obit_date'] = pd.to_datetime(df['data'], errors='coerce')
result['obituary_date'] = pd.to_datetime(result['obituary_date'], errors='coerce')
result['death_date'] = pd.to_datetime(result['death_date'], errors='coerce')
result['birth_date'] = pd.to_datetime(result['birth_date'], errors='coerce')
result['obit_vs_death_days'] = (result['obituary_date'] - result['death_date']).abs().dt.days

# fix death date, if close with obit, use death date, otherwise use obit date
result['death_date_final'] = pd.Series(None, index=result.index, dtype='datetime64[ns]')
result['death_date_final_source'] = pd.Series(None, index=result.index, dtype=object)
cond = result['obit_vs_death_days'] <= max_obit_vs_death_days_diff
result.loc[cond, 'death_date_final'] = result.loc[cond, 'death_date']
result.loc[cond, 'death_date_final_source'] = 'death_date val od'

result.loc[~cond, 'death_date_final'] = result.loc[~cond, 'obit_date']
result.loc[~cond, 'death_date_final_source'] = 'obit_date'

Last run: 2026-03-14 09:59:42


In [294]:
result['death_date_final_source'].value_counts()

death_date_final_source
death_date val od    28247
obit_date            11753
Name: count, dtype: int64

Last run: 2026-03-14 09:59:42


In [295]:
result['obit_vs_death_days'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count     29043.000000
mean        567.173364
std        5268.225140
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
90%           0.000000
95%           0.000000
99%       28607.960000
max      365243.000000
Name: obit_vs_death_days, dtype: float64

Last run: 2026-03-14 09:59:42


In [296]:
# validate birthdate
result['birth_date_final'] = pd.Series(None, index=result.index, dtype='datetime64[ns]')
result['birth_date_final_source'] = pd.Series(None, index=result.index, dtype=object)

# if birth vs death years is realistic, get the birth date as source
cond = result['death_date_final'].notnull() & result['birth_date'].notnull()
result.loc[cond, 'birth_vs_death_years'] = result.loc[cond, 'death_date_final'].dt.year - result.loc[cond, 'birth_date'].dt.year
cond = result['birth_vs_death_years'].between(1, max_birth_vs_death_years_diff)
result.loc[cond, 'birth_date_final'] = result.loc[cond, 'birth_date']
result['birth_date_final_source'] = 'birth_date val dd'

# if null get death date, sometimes it is captured as death instead of birth
result['temp_bd'] = result['birth_date'].fillna(result['death_date'])
result['birth_vs_death_years'] = result['death_date_final'].dt.year - result['temp_bd'].dt.year

# if resulting age is realistic, get the death date as source, use 2 min age because sometime
# death date years are just typo on year
cond = result['birth_vs_death_years'].between(2, max_birth_vs_death_years_diff) & result['birth_date_final'].isnull()
result.loc[cond, 'birth_date_final'] = result.loc[cond, 'temp_bd']
result.loc[cond, 'birth_date_final_source'] = 'inc_death_date'

result['birth_vs_death_years'] = result['death_date_final'].dt.year - result['birth_date_final'].dt.year

Last run: 2026-03-14 09:59:43


In [297]:
result['birth_date_final_source'].value_counts()

birth_date_final_source
birth_date val dd    39814
inc_death_date         186
Name: count, dtype: int64

Last run: 2026-03-14 09:59:52


In [298]:
#if age is validated against birth vs death years, use it as final age
result['age_final'] = pd.Series(None, index=result.index, dtype=float)
result['age_final_source'] = pd.Series(None, index=result.index, dtype=object)
result['age_vs_birth_death_years_diff'] = (result['age'] - result['birth_vs_death_years']).abs()

cond =result['age_vs_birth_death_years_diff']  <= 1
result.loc[cond, 'age_final'] = result.loc[cond, 'age']
result.loc[cond, 'age_final_source'] = 'age val vs bd'

Last run: 2026-03-14 10:00:01


In [308]:
#if age raw is higher, use it
cond = (result['age'] > result['birth_vs_death_years']) & result['age_final'].isnull()
result.loc[cond, 'age_final'] = result.loc[cond, 'age']
result.loc[cond, 'age_final_source'] = 'age raw higher'

#if birth vs age is higer, use it
cond = (result['birth_vs_death_years'] > result['age']) & result['age_final'].isnull()
result.loc[cond, 'age_final'] = result.loc[cond, 'birth_vs_death_years']
result.loc[cond, 'age_final_source'] = 'birth vs death years higher'

#else, keep age, source unvalidated
result['age_final'] = result['age_final'].fillna(result['age'])
result['age_final_source'] = result['age_final_source'].fillna('age unvalidated')

Last run: 2026-03-14 10:04:51


In [309]:
result['age_final_source'].value_counts()

age_final_source
age unvalidated                28930
age val vs bd                   9332
birth vs death years higher     1680
age raw higher                    58
Name: count, dtype: int64

Last run: 2026-03-14 10:04:59


In [310]:
result.groupby("age_final_source")['age_final'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
age_final_source,,,,,,,,,,,
age raw higher,58.0,62.189655,32.509355,3.0,27.25,75.5,86.0,93.3,96.3,116.3,122.0
age unvalidated,24418.0,57.387542,31.102716,1.0,25.00,68.0,84.0,91.0,94.0,99.0,121.0
age val vs bd,9332.0,75.052186,16.369205,1.0,66.00,79.0,87.0,92.0,95.0,99.0,110.0
birth vs death years higher,1680.0,78.401786,15.476775,10.0,71.00,82.0,89.0,94.0,97.0,101.0,105.0


Last run: 2026-03-14 10:05:51


In [299]:
result[result['age_vs_birth_death_years_diff'] > 1][['age','birth_vs_death_years']].describe()

,age,birth_vs_death_years
count,1738.000000,1738.000000
mean,18.366513,76.724396
std,15.218946,18.522683
min,1.000000,1.000000
25%,8.000000,69.000000
50%,16.000000,82.000000
75%,25.000000,89.000000
max,122.000000,105.000000


Last run: 2026-03-14 10:00:04


In [301]:
chosen = result[(result['age_vs_birth_death_years_diff'] > 1) & (result['birth_date_final_source']!="birth_date val dd")]

Last run: 2026-03-14 10:00:19


In [306]:
chosen['age_vs_birth_death_years_diff'].describe(percentiles=[0,0.01,0.03, 0.05, 0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1])

count     62.000000
mean      62.451613
std       21.291634
min        5.000000
0%         5.000000
1%         5.610000
3%        15.130000
5%        19.100000
10%       35.100000
20%       50.000000
30%       54.600000
40%       62.400000
50%       67.000000
60%       70.200000
70%       74.700000
80%       80.800000
90%       83.900000
100%     102.000000
max      102.000000
Name: age_vs_birth_death_years_diff, dtype: float64

Last run: 2026-03-14 10:02:16


In [304]:
chosen[['age', 'birth_vs_death_years', 'age_vs_birth_death_years_diff','birth_date_final','death_date_final','obit_date','death_date_raw','birth_date_final_source','death_date_final_source']]

,age,birth_vs_death_years,age_vs_birth_death_years_diff,birth_date_final,death_date_final,obit_date,death_date_raw,birth_date_final_source,death_date_final_source
182,9.0,63.0,54.0,1949-09-09,2012-03-07,2012-03-07,"September 9, 1949",inc_death_date,obit_date
3696,86.0,4.0,82.0,2007-04-15,2011-12-15,2011-12-15,April 2007,inc_death_date,obit_date
5456,28.0,78.0,50.0,1933-06-28,2011-10-30,2011-10-30,"June 28, 1933",inc_death_date,obit_date
5573,8.0,75.0,67.0,1936-06-08,2011-10-28,2011-10-28,"June 8, 1936",inc_death_date,obit_date
7050,98.0,15.0,83.0,1996-12-20,2011-09-20,2011-09-20,December 1996,inc_death_date,obit_date
...,...,...,...,...,...,...,...,...,...
38267,82.0,7.0,75.0,2002-01-26,2009-08-19,2009-08-19,"January 26, 2002",inc_death_date,obit_date
38550,71.0,2.0,69.0,2007-01-11,2009-08-11,2009-08-11,"January 11, 2007",inc_death_date,obit_date
38611,8.0,82.0,74.0,1927-08-06,2009-08-10,2009-08-10,08/06/1927,inc_death_date,obit_date
39323,81.0,7.0,74.0,2002-03-24,2009-07-20,2009-07-20,"March 24, 2002",inc_death_date,obit_date


Last run: 2026-03-14 10:01:08


In [307]:
df.loc[39323,"biografia"]

'Funeral services for Sara Cravey Bass, 81, of Ashburn, GA will be held, 11:00 AM, Thursday, July 23, 2009 at the Clements Chapel Methodist Church in Turner Co., GA.  Interment will follow in the church cemetery.  Mrs. Bass passed away Monday at Tift Regional Medical Center following a brief illness.\r The daughter of the late Thomas Watson and Martha Ford Cravey, Sr., she was born in Turner Co. and was a retired department store clerk.  She was a long time resident of Montgomery, AL before moving back to Turner Co. six years ago.  She was a graduate of Sycamore High School and was very involved in the Clements Chapel Methodist Church and enjoyed playing the piano since childhood.      She was the widow of Donald Ray Bass who passed away March 24, 2002.\r Mrs. Bass is survived by her son and daughter in law  Glenn Loran and Lina Suo Bass, Chattanooga, TN; her sisters, Helen Williford, Ashburn, GA and Jewel Forlaw, Greensboro, GA and a large number of nieces and nephews.\r The family wi

Last run: 2026-03-14 10:03:24
